<a href="https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Q1 — What decision does this improve?**
Prioritizing which specific web pages a content team should refresh first.

**Q2 — Who acts on the output, and what do they do?**
An SEO manager or editor. They take the top-K recommendations, review the page alongside the model's reason codes, and choose an action (metadata rewrite, full refresh, or just monitoring).

**Q3 — What does a wrong answer cost?**
- *False positive:* Wasted editorial bandwidth. Reviewing a page takes ~15 minutes; 10 bad recommendations waste 2.5 hours of human time.
- *False negative:* Missed opportunity. A page experiencing real decay goes unnoticed, resulting in compounding traffic losses.

**Q4 — Why does data or ML help?**
The signs of decay are hidden in complex interactions across numerous metrics (CTR, age, impressions, position). Manual rules fail to capture this complexity, which is why the ML model (Random Forest) triples the performance of the baseline heuristic.

### One-paragraph frame

> We are building **a ranked triage queue** to assist **content editors** in determining **which decaying pages to refresh first**. By analyzing **trailing-90-day search metrics**, we aim to score **the likelihood of a page currently declining**, optimizing for **Precision@50**. Making the wrong call leads to **wasted editorial time (false positives) or ignored traffic bleeding (false negatives)**. A basic ruleset is insufficient because **the true signal is buried in non-linear interactions across many features**. We will ensure all outputs are positioned strictly as **directional, decision-support tools**.

I am working within the Refresh / Content Opportunity Scoring lane.
Translated into an ML task, this is a Supervised Learning problem. Specifically, we are building a Binary Classification model, but we will use the outputted probabilities to create a Ranked Scoring System. We are not just sorting pages into "yes" or "no" buckets; we are scoring them from 0.0 to 1.0 so we can rank the queue from highest priority to lowest priority.

**Lane: Content Refresh and Opportunity Scoring**

**Task type: Scoring (producing a ranked queue).**

I am framing this as a scoring task because the goal is to generate a continuous priority value for each page, allowing us to sort the queue from highest to lowest risk. The end user (the editor) has a hard capacity limit (e.g., 50 pages), so they need an actionable ranking.

**Why the alternatives don't fit:**
- *Binary classification:* This would simply output a "yes/no" per page. If 5,000 pages get a "yes," the editor still doesn't know which one to tackle first.
- *Clustering:* This would group similar pages together, which is great for analysis but useless for a triage queue.
- *True ranking (Learning-to-Rank):* This requires pairwise preference data (page A is strictly worse than page B), which we don't naturally have here.

In [1]:
import pandas as pd

df = pd.read_csv('../../data/processed/model_predictions.csv')
score_col = 'best_model_probability'
print(f'Total scored rows: {len(df):,}')
print(f'Score distribution: {df[score_col].min():.4f} to {df[score_col].max():.4f}')
print(f'Median priority score: {df[score_col].median():.4f}')
print()

# Examining the top K thresholds based on the continuous score
for pct in [10, 25, 50, 100, 200]:
    n = min(pct, len(df))
    top_k = df.nlargest(n, score_col)
    true_decline_rate = top_k.is_declining_label.mean()
    print(f'Top {n:>3} Queue: true decline rate = {true_decline_rate:.3f}  (overall base rate = 0.542)')

Total scored rows: 30,000
Score distribution: 0.0021 to 0.8518
Median priority score: 0.5381

Top  10 Queue: true decline rate = 1.000  (overall base rate = 0.542)
Top  25 Queue: true decline rate = 1.000  (overall base rate = 0.542)
Top  50 Queue: true decline rate = 1.000  (overall base rate = 0.542)
Top 100 Queue: true decline rate = 1.000  (overall base rate = 0.542)
Top 200 Queue: true decline rate = 0.990  (overall base rate = 0.542)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Current target:** `is_declining_label` (maps to 1 if `trend_direction == "down"`, else 0).

This is fundamentally a **proxy label**. It is calculated by comparing the most recent 30 days of impressions against the prior 30 days within a closed 90-day window. Thus, it identifies pages that are *currently* declining, rather than predicting what will happen in the future.

It serves as a strong proxy because:
- Over half the dataset (54.2%) meets the decline criteria, providing rich training signal.
- The definition is deterministic and reproducible.
- We deliberately exclude `trend_direction` and `trend_pct` from the feature set to prevent data leakage.

**Future improvement (capstone trajectory):** The ultimate goal is to swap this proxy for a true predictive target. By leveraging the full daily warehouse, we can use features from an older 90-day window to predict decline in the *subsequent* 30-day window.

In [2]:
raw = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
raw['is_declining_label'] = (raw.trend_direction.str.lower() == 'down').astype(int)

print('Target Label Distribution:')
print(raw['is_declining_label'].value_counts().to_string())
print(f'\nOverall Decline Rate: {raw.is_declining_label.mean():.3f}')
print(f'Total observations: {len(raw):,}\n')

print('Proxy Label Logic: trend_direction (derived from trend_pct)')
print('  down:   last 30d impressions are >= 20% lower than prev 30d')
print('  up:     last 30d impressions are >= 20% higher than prev 30d')
print('  flat:   zero impressions in both windows')
print('  stable: falls within the +/- 20% bounds')
print('  new:    prev 30d was 0, but last 30d has > 0')

Target Label Distribution:
is_declining_label
1    16262
0    13738

Overall Decline Rate: 0.542
Total observations: 30,000

Proxy Label Logic: trend_direction (derived from trend_pct)
  down:   last 30d impressions are >= 20% lower than prev 30d
  up:     last 30d impressions are >= 20% higher than prev 30d
  flat:   zero impressions in both windows
  stable: falls within the +/- 20% bounds
  new:    prev 30d was 0, but last 30d has > 0


In [3]:
import os, sys, subprocess
import pandas as pd

# 1. Colab Directory Fix (Ensuring we are in the right folder)
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-internship-assignment1"
if IN_COLAB and os.path.basename(os.getcwd()) != "notebooks":
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Basil-Maqbool/flyrank-internship-assignment1", REPO_DIR], check=True)
    os.chdir(f"{REPO_DIR}/work/notebooks")

# 2. Load the dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# 3. Define the Target Proxy
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

# 4. Display the Unit of Analysis
print(f"Total rows (pages) in dataset: {len(df):,}")
print("Unit of Analysis: One row equals one unique content page (URL).")
print("Here is a slice showing the grain and our target column:")

# Displaying 3 rows to prove the grain
display(df[['content_id', 'impressions_90d', 'content_age_days', 'trend_direction', 'is_declining_label']].head(3))

Total rows (pages) in dataset: 30,000
Unit of Analysis: One row equals one unique content page (URL).
Here is a slice showing the grain and our target column:


,content_id,impressions_90d,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,3803,187,down,1
1,content_a1fb4e703a9e,15320,445,down,1
2,content_9aa793d4d895,12581,141,down,1


**Primary metric: Precision@50**

Precision@K answers a highly specific business question: *Out of the top 50 pages the model recommends, how many are actually suffering a decline?*

This perfectly mirrors the workflow of a content team. If an editor can only review 50 pages a week, Precision@50 tells us exactly how reliable that 50-page batch will be.

**Why other metrics fall short:**
- *Recall:* Identifying 100% of all decaying pages is pointless if the team only has the bandwidth to fix 50 of them.
- *ROC-AUC:* This evaluates the model's ranking ability across the entire distribution (from rank 1 to rank 16,000). We only care about the extreme head of the queue (the top 50).

A fixed rule (e.g., "refresh if age > 180 days AND impressions > 500") is rigid and brittle. It treats a 181-day-old page vastly differently than a 179-day-old page, creating artificial cutoffs. Furthermore, a fixed rule cannot weigh complex feature interactions. For example, a 300-day-old page holding position 1 might be fine, but a 100-day-old page rapidly losing CTR needs immediate attention. Machine Learning is necessary here because it can dynamically capture these non-linear relationships across multiple variables (age, position, clicks, content type) to find the true signal of decay without relying on human guesswork.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

1- Task type named (Classification / Ranking).

2- Target/proxy defined (is_declining_label).

3- Success metric chosen (Precision@50) and tied to editorial capacity.

4- Code cell executed showing a real dataframe where one row = one page.

5- Explained why ML captures non-linear relationships better than fixed rules.